# 1 - Imports

In [1]:
%reload_ext autoreload
%autoreload 2

In [2]:
import sys
from pathlib import Path

sys.path.append(str(Path().resolve().parents[0]))

In [3]:
import pandas as pd
import numpy as np

In [4]:
from src.utils import config, io
from src.features import selection, pruning

/Users/hippolytegrandet/Desktop/Dev/country_risk_rating/.venv/lib/python3.9/site-packages/pypdf/_crypt_providers/_cryptography.py:32: CryptographyDeprecationWarning: ARC4 has been moved to cryptography.hazmat.decrepit.ciphers.algorithms.ARC4 and will be removed from cryptography.hazmat.primitives.ciphers.algorithms in 48.0.0.
  from cryptography.hazmat.primitives.ciphers.algorithms import AES, ARC4


# 2 - Feature Selection (Independent of Target)

In [5]:
dataset = io.load_csv(config.INTERIM_DATA_DIR / 'merged_dataset.csv', index_col=0)
dataset

,YEAR,ISO3_COUNTRY_CODE,OECD_RATING,GEO_REGION,ADMIN_REGION,LENDING_TYPE,INCOME_GROUP,NY.GDP.MKTP.CD,NY.GDP.MKTP.KD.ZG,NY.GDP.MKTP.PP.CD,...,SP.POP.GROW,SP.URB.TOTL.IN.ZS,EN.POP.DNST,SE.PRM.CUAT.ZS,SP.POP.DPND,SL.TLF.CACT.ZS,IQ.CPA.BREG.XQ,IQ.CPA.PADM.XQ,FS.AST.PRVT.GD.ZS,FM.AST.PRVT.GD.ZS
COUNTRY_PERIOD_INDEX,,,,,,,,,,,,,,,,,,,,,
AFG-1999,1999,AFG,7,MEA,MNA,IDX,LIC,NaN,NaN,NaN,...,3.728116,18.390327,30.491981,NaN,108.686031,46.609,NaN,NaN,NaN,NaN
AFG-2000,2000,AFG,7,MEA,MNA,IDX,LIC,3.521418e+09,NaN,1.637703e+10,...,1.212176,18.558200,30.863847,NaN,109.586048,46.562,NaN,NaN,NaN,NaN
AFG-2001,2001,AFG,-,MEA,MNA,IDX,LIC,2.813572e+09,-9.431974,1.516633e+10,...,0.762005,18.741238,31.099929,NaN,110.219341,46.526,NaN,NaN,NaN,NaN
AFG-2002,2002,AFG,-,MEA,MNA,IDX,LIC,3.825701e+09,28.600001,1.980700e+10,...,5.252030,18.941248,32.776961,NaN,110.534611,46.505,NaN,NaN,NaN,NaN
AFG-2003,2003,AFG,-,MEA,MNA,IDX,LIC,4.520947e+09,8.832278,2.198200e+10,...,6.145194,19.160035,34.854344,NaN,110.557540,46.497,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
ZWE-2022,2022,ZWE,7,SSF,SSA,IDB,LMC,4.075756e+10,6.139263,8.670632e+10,...,1.706209,38.705521,41.538209,NaN,82.547605,65.214,3.0,3.0,5.826323,5.826323
ZWE-2023,2023,ZWE,7,SSF,SSA,IDB,LMC,3.587178e+10,5.350869,9.463284e+10,...,1.677096,39.291937,42.240719,NaN,81.625352,67.786,3.0,3.0,6.477697,6.477697
ZWE-2024,2024,ZWE,7,SSF,SSA,IDB,LMC,4.153941e+10,1.742395,9.860990e+10,...,1.780482,39.893372,NaN,NaN,80.100412,67.725,3.0,3.0,NaN,NaN


In [6]:
selection_params = {
    'max_missing_ratio': 0.6,
    'low_var_threshold': 0.001,
    'max_corr': 0.95,
    'top_k_mi': 100
}

In [9]:
dataset = selection.get_dataset_feature_selected(
    dataset, 
    selection_params, 
    verbose=True
)

Initial Dataset Shape: (5025, 85)
Drop Null Target Shape: (4204, 85)
Filter Missingness Shape: (4204, 79)
Filter Low Variance Shape: (4204, 78)
Filter Correlated Shape: (4204, 64)
Final Dataset Shape: (4204, 64)


In [7]:
dataset = selection.get_dataset_feature_selected(
    dataset, 
    selection_params, 
    verbose=True
)

Initial Dataset Shape: (5628, 81)
Drop Null Target Shape: (4719, 81)
Filter Missingness Shape: (4719, 75)
Filter Low Variance Shape: (4719, 75)
{'max_missing_ratio': 0.6, 'low_var_threshold': 0.001, 'max_corr': 0.95, 'top_k_mi': 100}
Final Dataset Shape: (4719, 61)


In [8]:
io.save_csv(dataset, config.INTERIM_DATA_DIR / 'feature_selected_dataset.csv', index=True)

# 3 - Feature Pruning (Aligned with Target)

In [9]:
dataset = io.load_csv(config.INTERIM_DATA_DIR / 'feature_selected_dataset.csv', index_col=0)
dataset

,YEAR,ISO3_COUNTRY_CODE,OECD_RATING,GEO_REGION,ADMIN_REGION,LENDING_TYPE,INCOME_GROUP,NY.GDP.MKTP.CD,NY.GDP.MKTP.KD.ZG,NY.GDP.MKTP.PP.CD,...,SL.UEM.TOTL.ZS,EN.URB.LCTY.UR.ZS,SP.POP.TOTL,SP.POP.GROW,SP.URB.TOTL.IN.ZS,EN.POP.DNST,SE.PRM.CUAT.ZS,SP.POP.DPND,SL.TLF.CACT.ZS,FS.AST.PRVT.GD.ZS
COUNTRY_PERIOD_INDEX,,,,,,,,,,,,,,,,,,,,,
AFG-1999,1999,AFG,7,MEA,MNA,IDX,LIC,NaN,NaN,NaN,...,7.877,62.832635,19887785.0,3.728116,18.390327,30.491981,NaN,108.686031,46.609,NaN
AFG-2000,2000,AFG,7,MEA,MNA,IDX,LIC,3.521418e+09,NaN,1.637703e+10,...,7.897,64.272506,20130327.0,1.212176,18.558200,30.863847,NaN,109.586048,46.562,NaN
AFG-2008,2008,AFG,7,MEA,MNA,IDX,LIC,1.010930e+10,3.924984,3.532112e+10,...,7.879,55.947738,26482622.0,2.186546,21.124148,40.603195,NaN,106.334376,46.682,9.388328
AFG-2009,2009,AFG,7,MEA,MNA,IDX,LIC,1.241615e+10,21.390528,4.314095e+10,...,7.808,53.859058,27466101.0,3.646381,21.688548,42.111067,NaN,104.799467,46.746,10.584131
AFG-2010,2010,AFG,7,MEA,MNA,IDX,LIC,1.585667e+10,14.362441,4.993663e+10,...,7.809,52.235805,28284089.0,2.934687,22.261480,43.365207,NaN,103.144456,46.815,11.575044
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
ZWE-2022,2022,ZWE,7,SSF,SSA,IDB,LMC,4.075756e+10,6.139263,8.670632e+10,...,10.087,25.045614,16069056.0,1.706209,38.705521,41.538209,NaN,82.547605,65.214,5.826323
ZWE-2023,2023,ZWE,7,SSF,SSA,IDB,LMC,3.587178e+10,5.350869,9.463284e+10,...,9.348,24.579040,16340822.0,1.677096,39.291937,42.240719,NaN,81.625352,67.786,6.477697
ZWE-2024,2024,ZWE,7,SSF,SSA,IDB,LMC,4.153941e+10,1.742395,9.860990e+10,...,9.435,24.159103,16634373.0,1.780482,39.893372,NaN,NaN,80.100412,67.725,NaN


In [10]:
from src.features import engineering

dataset = engineering.create_features(dataset)
print(f"After feature engineering: {dataset.shape}")
print("New features:", [c for c in dataset.columns if c.startswith("ENG_")])

After feature engineering: (4719, 66)
New features: ['ENG_interest_rate_spread', 'ENG_gdp_per_capita_growth', 'ENG_net_fdi_pct_gdp', 'ENG_external_debt_burden', 'ENG_savings_investment_gap']


In [11]:
dataset = selection.filter_sparse_rows(dataset, max_missing_ratio=0.5)
print(f"After row filter: {dataset.shape}")

After row filter: (4238, 66)


In [12]:
X = dataset.drop(columns=['OECD_RATING']) # Drop Target
X = X.drop(columns=['ISO3_COUNTRY_CODE']) # Drop Country ID (maybe YEAR)
y = dataset['OECD_RATING']

In [13]:
io.save_csv(X, config.PROCESSED_DATA_DIR / 'X.csv', index=True)

In [14]:
io.save_csv(y, config.PROCESSED_DATA_DIR / 'y.csv', index=True)